# EFS-03 resource addendum

This additive notebook preserves the original notebook and adds source-backed controls for futures roll strategies.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['efs03_roll_formula_reference.csv','efs03_futures_data_hygiene.csv','efs03_gold_hedge_reference.csv','efs03_timestamp_alignment_demo.csv','efs03_sharpe_funding_cases.csv','efs03_back_adjustment_demo.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Roll formulas and causal framework

Exact arbitrage formulas apply to ES and FX futures; commodities usually require estimating roll from the forward curve.

In [ ]:
roll = data['efs03_roll_formula_reference.csv']
print(roll[['instrument','roll_formula','example_roll']].to_string(index=False))
assert roll['instrument'].str.contains('E-mini').any()
assert roll['roll_formula'].str.contains('slope of log futures').any()

## 2. Data hygiene controls

The transcript adds implementation details: continuous futures need back-adjustment, paired markets need matching timestamps, and spot hedges must not cancel the roll edge.

In [ ]:
hygiene = data['efs03_futures_data_hygiene.csv']
print(hygiene[['control','correct_action']].to_string(index=False))
assert hygiene['control'].str.contains('Timestamp').any()
assert hygiene['control'].str.contains('Back-adjusted').any()

## 3. Spot hedge selection

GLD is a direct enough gold spot proxy for the GC exercise. USO is not a good crude spot hedge for roll harvesting because it holds futures. XLE is only a partial proxy because it adds equity/company risks.

In [ ]:
hedges = data['efs03_gold_hedge_reference.csv']
print(hedges.to_string(index=False))
assert hedges.query("spot_proxy == 'USO'")['works'].iloc[0] == 'no'
assert hedges.query("spot_proxy == 'GLD'")['works'].iloc[0] == 'yes'

## 4. Timestamp mismatch and back-adjustment demos

These are small synthetic demonstrations of the two backtest bugs described in the transcript.

In [ ]:
ts = data['efs03_timestamp_alignment_demo.csv']
backadj = data['efs03_back_adjustment_demo.csv']
print(ts[['mismatched_spread','aligned_spread']].agg(['mean','std']).round(4))
print(backadj[['day','stitched_price','back_adjusted_price','return_label']].to_string(index=False))
assert abs(ts['mismatched_spread'].mean() - ts['aligned_spread'].mean()) > 0.01
assert 'not real P&L' in set(backadj['return_label'])

## 5. Sharpe funding treatment

The risk-free rate is not always subtracted. The funding treatment depends on whether cash remains available or is spent on the hedge.

In [ ]:
sharpe = data['efs03_sharpe_funding_cases.csv']
print(sharpe.to_string(index=False))
assert sharpe.query("strategy_case == 'Short GC future plus long GLD'")['subtract_risk_free'].iloc[0] == 'yes'